# Weekly Unit Commitment + Transmission Congestion (Simplified DC Power Flow)

The "supply and demand operations department" of a power company determines the start-up/shut-down and output allocation of generating units for the following week (tens to 168 hours). Each unit is connected to a specific bus (node in the transmission grid), and its output is **coupled to the power flow constraints of all transmission lines through a simplified DC power flow (PTDF: Power Transfer Distribution Factor)**. The start-up decision of a single unit ripples through the economics of all other units via the congestion of distant transmission lines — this is not an approximation, but the exact linear coupling standardized in ISO/RTO practices (market clearing, congestion management).

Business meaning of each constraint:

- **Start-up/shut-down logic, minimum up/down time, ramping**: Thermal units cannot rapidly change output or frequently start/stop (thermal stress on turbines, combustion stability).
- **Reserves (spinning reserve)**: In preparation for sudden demand changes or unit trips, the total maximum output of online units must equal or exceed the actual supply plus a safety margin.
- **Simplified DC power flow (PTDF)**: The power flow of each transmission line is determined by a "linear combination of net power injections across all buses" and must not exceed upper and lower limits of thermal capacity.
- **Unplanned outage (load shedding) backstop**: Allows for extremely high-cost unplanned outages if demand simply cannot be met due to transmission congestion or supply shortages (ensuring feasibility).

The difficulty of this problem is not the weakness of non-linearities, but its **combinatorics as a large-scale mixed-integer linear programming (MILP) problem** (start-up/shut-down $\times$ tight coupling of PTDF $\times$ weekly multi-period). Subject: `samples/energy_and_microgrid/weekly_uc_ramp.py`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/energy_and_microgrid"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import weekly_uc_ramp as uc
from pyscipopt import Model, quicksum

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

## 1. Naive Formulation

The default scale is 15 units $\times$ 8 buses $\times$ 10 transmission lines $\times$ 48 hours. This problem is **linear** (zero non-linear constraints), and its difficulty lies in the scale of the combination of start-up/shut-down (binary) $\times$ tight coupling of PTDF $\times$ multi-period.

In [ ]:
m0 = uc.build_model("default")
print(f"変数 {m0.getNVars()}(うちバイナリ {m0.getNBinVars()})、制約 {m0.getNConss()}")
print("非線形制約:", sum(1 for c in m0.getConss() if c.isNonlinear()), "件(この問題は純粋MILP)")
kinds = {}
for c in m0.getConss():
    k = c.name.rsplit("_", 1)[0] if "_" in c.name else c.name
    kinds[k] = kinds.get(k, 0) + 1
pd.Series(kinds).sort_values(ascending=False).head(10)
del m0

`flow_ub`/`flow_lb` (transmission line thermal capacity constraints) are the most numerous — there are as many as (transmission lines $\times$ periods), and each constraint features a **linear combination spanning all buses**: $\sum_{\text{bus}} \mathrm{PTDF}[l,\text{bus}]\cdot(\text{generation}-\text{demand}+\text{shedding})$.
This is the true nature of the tight coupling where "the start-up decision of a single unit ripples through to all units via distant transmission lines."

## 2. Diagnosis

In [ ]:
report = mk.analyze(lambda: uc.build_model("default"), name="weekly-uc-ramp",
                    time_limit=30)
print(report.summary())
print("\ngap:", f"{report.metrics.get('gap', 0):.2%}", "/ ノード数:", report.metrics.get("nodes"))

At this scale, the gap is nearly 0% (SCIP's presolve/heuristics instantly find a near-optimal solution near the root). Only `numerical_scale` (order of magnitude difference in coefficient scales remaining after presolve) triggers — in terms of the gap, it is "not difficult", but the broadness of the coefficient range itself remains. Let's look inside.

## 3. Looking Inside the Diagnosis — Coefficient Scale and Condition Number

We check the remaining Big-M candidates pointed out by `numerical_scale` directly with `static_diag`, and also measure the condition number ($\kappa(A)$) as a linear program.

In [ ]:
from minlpkit.collectors.static_diag import residual_scale, matrix_condition, scip_basis_condition

m1 = uc.build_model("default")
rs = residual_scale(m1)
print(f"presolve後の係数比 = {rs['ratio']:.3g}(中央値 {rs['median']:.3g})")
bigm_df = pd.DataFrame(rs["bigm"])
bigm_df

In [ ]:
fig = go.Figure(layout=base_layout(
    "presolve後に残る大係数(Big-M候補、上位10件)",
    "制約/変数名", "係数の絶対値(対数軸)", height=420))
fig.add_trace(go.Bar(x=bigm_df["name"], y=bigm_df["magnitude"],
    marker_color=[C["warn"] if s == "制約係数" else C["s1"] for s in bigm_df["source"]],
    text=bigm_df["source"], textposition="outside"))
fig.update_yaxes(type="log")
fig.update_xaxes(tickangle=-30)
show(fig)

At the top are `reserve_*` (spinning reserve: $\sum \mathrm{pmax}\cdot u \ge \mathrm{served}\cdot(1+\mathrm{reserve})$, where the coefficient `pmax` is on the order of tens to hundreds) and `su_cost` (start-up cost, hundreds to 1500) related constraints. These are natural values as physical quantities and costs, and not truly "unnecessarily loose Big-Ms" — since the linked constraints `p<=pmax*u`/`p>=pmin*u` (linking unit output upper/lower limits to on/off status) take a Big-M form, we will convert this part to Indicator and confirm the effect.

## 4. Trying Improvements — Replacing Output $\leftrightarrow$ on/off Linked Constraints with Indicators

`p[i,t] <= pmax[i]*u[i,t]` and `p[i,t] >= pmin[i]*u[i,t]` express the logic "output is 0 if off, `[pmin,pmax]` if on" using Big-M (`pmax`/`pmin`). Since `weekly_uc_ramp.py` has no arguments to switch variants, we will assemble a variant within this notebook that replicates the same construction logic as `build_model` but replaces only these two constraints with `addConsIndicator` (since the variable bound `ub=pmax[i]` for `p` remains intact, we only need one Indicator for the off-side `p<=0` and an Indicator for just the lower bound of the on-side).

In [ ]:
def build_indicator(scale="default"):
    '''p<=pmax*u / p>=pmin*u (Big-M) を addConsIndicator に置き換えた変種。'''
    d = uc._data(scale)
    nU, nB, nL, nT = d["nU"], d["nB"], d["nL"], d["nT"]
    edges, PTDF, cap = d["edges"], d["PTDF"], d["cap"]
    pmin, pmax = d["pmin"], d["pmax"]
    a_cost, b_cost, su_cost = d["a_cost"], d["b_cost"], d["su_cost"]
    ramp, mu, md, bus_of = d["ramp"], d["mu"], d["md"], d["bus_of"]
    demand = d["demand"]

    m = Model("Weekly_UC_Ramp_Network_Indicator")
    U, B, L, T = range(nU), range(nB), range(nL), range(nT)
    u, v, w, p, fcv = {}, {}, {}, {}, {}
    for i in U:
        for t in T:
            u[i, t] = m.addVar(vtype="B", name=f"u_{i}_{t}")
            v[i, t] = m.addVar(vtype="B", name=f"v_{i}_{t}")
            w[i, t] = m.addVar(vtype="B", name=f"w_{i}_{t}")
            p[i, t] = m.addVar(vtype="C", lb=0.0, ub=float(pmax[i]), name=f"p_{i}_{t}")
            fcv[i, t] = m.addVar(vtype="C", lb=0.0, name=f"fc_{i}_{t}")
    shed = {(b_, t): m.addVar(vtype="C", lb=0.0, ub=float(demand[b_, t]), name=f"shed_{b_}_{t}")
            for b_ in B for t in T}
    for i in U:
        for t in T:
            # off (u=0) なら出力0。on (u=1) のときだけ下限pminを課す(上限は変数境界ub=pmaxで既に厳密)
            m.addConsIndicator(p[i, t] <= 0, binvar=u[i, t], activeone=False, name=f"ind_off_{i}_{t}")
            m.addConsIndicator(-p[i, t] <= -float(pmin[i]), binvar=u[i, t], activeone=True,
                               name=f"ind_min_{i}_{t}")
            prev = u[i, t - 1] if t > 0 else 0
            m.addCons(v[i, t] - w[i, t] == u[i, t] - prev)
            m.addCons(v[i, t] + w[i, t] <= 1)
            if t > 0:
                m.addCons(p[i, t] - p[i, t - 1] <= float(ramp[i]) * u[i, t - 1] + float(pmin[i]) * v[i, t])
                m.addCons(p[i, t - 1] - p[i, t] <= float(ramp[i]) * u[i, t] + float(pmax[i]) * w[i, t])
            for tau in range(t + 1, min(t + int(mu[i]), nT)):
                m.addCons(u[i, tau] >= v[i, t])
            for tau in range(t + 1, min(t + int(md[i]), nT)):
                m.addCons(u[i, tau] <= 1 - w[i, t])
            m.addCons(fcv[i, t] >= float(a_cost[i]) * u[i, t] + float(b_cost[i]) * p[i, t])
    for t in T:
        m.addCons(quicksum(p[i, t] for i in U) + quicksum(shed[b_, t] for b_ in B)
                  == float(demand[:, t].sum()), name=f"balance_{t}")
        served = float(demand[:, t].sum()) - quicksum(shed[b_, t] for b_ in B)
        m.addCons(quicksum(float(pmax[i]) * u[i, t] for i in U)
                  >= served * (1 + uc.RESERVE), name=f"reserve_{t}")
    gen_at_bus = {(b_, t): quicksum(p[i, t] for i in U if int(bus_of[i]) == b_)
                  for b_ in B for t in T}
    for l in L:
        for t in T:
            net = quicksum(float(PTDF[l, b_]) * (gen_at_bus[b_, t] - float(demand[b_, t]) + shed[b_, t])
                           for b_ in B)
            m.addCons(net <= float(cap[l]), name=f"flow_ub_{l}_{t}")
            m.addCons(net >= -float(cap[l]), name=f"flow_lb_{l}_{t}")
    obj = quicksum(fcv[i, t] + float(su_cost[i]) * v[i, t] for i in U for t in T)
    obj += quicksum(uc.VOLL * shed[b_, t] for b_ in B for t in T)
    m.setObjective(obj, "minimize")
    m.data = dict(u=u, p=p, shed=shed, scale=scale, dims=(nU, nB, nL, nT))
    return m

mb = uc.build_model("small"); mb.hideOutput(); mb.optimize()
mi = build_indicator("small"); mi.hideOutput(); mi.optimize()
print(f"baseline  : obj={mb.getObjVal():.2f}  status={mb.getStatus()}")
print(f"indicator : obj={mi.getObjVal():.2f}  status={mi.getStatus()}")

It reaches the same optimal value at the small scale (matching within floating-point error). At the default scale, as seen in Section 2, the gap is already almost 0%, so improvements are hard to see in `compare_variants`'s `final_gap`/`nodes`. Here, in addition, we also compare the **condition number $\kappa(A)$** (static, SVD before presolve) and the **SCIP basis condition number** (the optimal basis actually solved, `getCondition()`).

In [ ]:
df = mk.compare_variants(
    {"baseline (Big-M)": lambda: uc.build_model("default"),
     "indicator":        lambda: build_indicator("default")},
    time_limit=30)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

In [ ]:
kappa_base = matrix_condition(uc.build_model("default"))
kappa_ind = matrix_condition(build_indicator("default"))
basis_base = scip_basis_condition(uc.build_model("default"))
basis_ind = scip_basis_condition(build_indicator("default"))
print(f"静的 κ(A)     : baseline={kappa_base['kappa']:.3g}  indicator={kappa_ind['kappa']:.3g}")
print(f"SCIP基底κ     : baseline={basis_base}  indicator={basis_ind}")

In [ ]:
base = df.loc[df["variant"] == "baseline (Big-M)"].iloc[0]
ind  = df.loc[df["variant"] == "indicator"].iloc[0]
labels = ["baseline (Big-M)", "indicator"]
bar_colors = [C["muted"], C["s1"]]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界", "探索ノード数", "静的 κ(A)(低いほど良い)"))
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [base["root_dual"], ind["root_dual"]], lambda v: f"{v:.0f}")
add_bars(2, [base["nodes"], ind["nodes"]], lambda v: f"{int(v)}")
add_bars(3, [kappa_base["kappa"], kappa_ind["kappa"]], lambda v: f"{v:.2e}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="p<=>u の Indicator化 before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)
print(f"root_dual : {base['root_dual']:.1f} -> {ind['root_dual']:.1f}")
print(f"final_gap : {base['final_gap']:.4%} -> {ind['final_gap']:.4%}")
print(f"nodes     : {int(base['nodes'])} -> {int(ind['nodes'])}")

**Honest verification result**: `root_dual`/`final_gap`/`nodes` are already almost optimal from baseline (nodes 1, gap 0.02%), and even with Indicatorization, these metrics hardly budge (nodes stay at 1, final_gap actually worsens slightly from 0.02% to 0.26%) —— the room for improvement itself is already filled by presolve/heuristics.

**A further surprising empirical result**: The static $\kappa(A)$ **worsened by nearly 600 times**, going from baseline = 4.6e4 to indicator = 2.76e7, and the SCIP basis $\kappa$ (getCondition()) also slightly worsened from 235171 to 260790. Because Indicator constraints are maintained internally in SCIP as linear constraints via "slack variables that toggle constraints on and off," static coefficient matrix extraction (linear constraint coefficients picked up by `extract_coefficients`) can **introduce different scale differences even though Big-M was supposedly eliminated** —— this empirical measurement defies the intuition that "eliminating Big-M = always numerically sounder," confirming that the effects of Indicatorization must be measured on a case-by-case basis. **The `numerical_scale` in this subject does not imply an "empirical difficulty" but simply points to the natural structure that "generation costs, output capacities, and start-up costs inherently possess orders of magnitude different scales."** —— Similar to the lesson learned in the petroleum_pooling notebook, the diagnosis picking up a symptom and that symptom affecting actual solving performance are on different axes. The true difficulty of this problem is not the weakness of non-linearities, but **the combinatorics of dense coupling of the entire transmission network via PTDF $\times$ weekly multi-period $\times$ tens of units**, and if one wants to reduce the gap, decompositions (Benders/Column Generation) or tuning could be a more appropriate approach.

## Summary

- Although this problem is a **pure MILP with no non-linearities**, the scale of the combination of grid coupling via PTDF and weekly multi-periods itself is the source of its difficulty. At the default scale, SCIP's presolve/heuristics are strong and reach near-optimality in 30 seconds, so the effect of strengthening relaxations like eliminating Big-M is practically invisible empirically.
- Practically, this reflects the supply-demand operation of "which units to start when, and how to allocate output within the thermal capacity constraints of the transmission grid," encompassing the obligation to maintain not just single-bus supply-demand balances but thermal capacity constraints per transmission line (market clearing, congestion management).
- Lesson: Even if a diagnostic rule is triggered, whether it "actually hinders solving" must always be checked with `compare_variants`. In this subject, `numerical_scale` was triggered, but addressing it did not move the gap/node count — diagnosis is a starting point, and only holds meaning when paired with effect measurement.
- Further unexpected lesson: While Indicatorization did not worsen `root_dual`/`final_gap`/`nodes`, the static $\kappa(A)$ and SCIP basis $\kappa$ **actually worsened** (see Section 4). The intuition that "eliminating Big-M always leads to numerical soundness" was disproven in this subject — because Indicator constraints are maintained as linear constraints via slack variables inside SCIP, the appearance of the coefficient scale itself can change.

Related: [Method Guide 3. Eliminating Big-M](../../playbook/03-bigm.md) /
[Method Guide 8. Condition Number and Numerical Soundness](../../playbook/08-condition.md) /
API [`mk.analyze`](../../api/pipeline.md)